In [1]:
import os
import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import timm

# ---------- 配置 ----------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = r"D:\oil_spill_detect\best_models\best_student.pth"     # 替换为你的模型权重路径
SAMPLE_DIR = r"D:\oil_spill_detect\Samples"                          # 替换为你的 Samples 文件夹路径
MODEL_NAME = "mobilenetv3_small_100"                                 # 学生模型架构
NUM_CLASSES = 2                                                      # 二分类

# ---------- 图像预处理 ----------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ---------- 加载模型 ----------
def load_model(weights_path, device):
    model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES)
    state_dict = torch.load(weights_path, map_location=device)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model

model = load_model(MODEL_PATH, DEVICE)
print(f"模型已加载：{MODEL_PATH}")

# ---------- 预测单张图像 ----------
def predict_image(image_path, model, transform, device):
    # 读取图像，转换为 RGB（模型需要三通道）
    img = Image.open(image_path).convert('L')   # 确保灰度
    img = img.convert('RGB')                   # 转三通道
    img_tensor = transform(img).unsqueeze(0)   # 添加 batch 维度
    img_tensor = img_tensor.to(device)

    with torch.no_grad():
        logits = model(img_tensor)
        probs = F.softmax(logits, dim=1).cpu().numpy()[0]
        pred_class = int(probs.argmax())
        confidence = probs[pred_class]

    return pred_class, confidence, probs

# ---------- 批量预测 Samples 文件夹中的图像 ----------
if __name__ == "__main__":
    # 支持的图像扩展名
    exts = ('.png', '.jpg', '.jpeg', '.tif', '.tiff')
    image_files = [f for f in os.listdir(SAMPLE_DIR) if f.lower().endswith(exts)]

    if not image_files:
        print(f"未在 {SAMPLE_DIR} 中找到图像文件。")
    else:
        print(f"找到 {len(image_files)} 张图像，开始预测...\n")
        for img_file in image_files:
            img_path = os.path.join(SAMPLE_DIR, img_file)
            pred_class, conf, probs = predict_image(img_path, model, transform, DEVICE)
            label = "Oil Spill" if pred_class == 1 else "No Oil Spill"
            print(f"{img_file:20s} -> {label:12s} (confidence: {conf:.4f}) | probs: [NoOil={probs[0]:.3f}, Oil={probs[1]:.3f}]")

模型已加载：D:\oil_spill_detect\best_models\best_student.pth
找到 8 张图像，开始预测...

0_0_0_img_5kPtgTDfaBqbQJtE_ADR_cls_0.jpg -> No Oil Spill (confidence: 0.9910) | probs: [NoOil=0.991, Oil=0.009]
0_0_0_img_72gUvSnnkPTMGLF3_GBR_cls_0.jpg -> No Oil Spill (confidence: 0.9722) | probs: [NoOil=0.972, Oil=0.028]
0_0_0_img_ca9QOv0U0q6eU9Ow_PHI_cls_0.jpg -> No Oil Spill (confidence: 0.9332) | probs: [NoOil=0.933, Oil=0.067]
0_0_0_img_ky8UpY0ljZcNZVw3_GBR_cls_1.jpg -> Oil Spill    (confidence: 1.0000) | probs: [NoOil=0.000, Oil=1.000]
0_0_0_img_qBIhHv0a36ynbvIW_GIB_cls_1.jpg -> Oil Spill    (confidence: 0.9999) | probs: [NoOil=0.000, Oil=1.000]
0_0_0_img_QU4PTDLBV3dBxN7b_PHI_cls_0.jpg -> No Oil Spill (confidence: 0.9436) | probs: [NoOil=0.944, Oil=0.056]
1_200_0_img_9c3d5585_TRI_cls_1.jpg -> Oil Spill    (confidence: 0.9976) | probs: [NoOil=0.002, Oil=0.998]
1_200_0_img_dtsR0emPVPPNdxSS_SFr_cls_1.jpg -> Oil Spill    (confidence: 0.8373) | probs: [NoOil=0.163, Oil=0.837]
